In [1]:
# Milestone 7: prepare the trained models for serving in the web app. Three things the app
# needs that the training notebooks did not leave behind: (1) the imputer + scaler the neural
# network was trained with (fitted inside notebook 05 but never saved), (2) a small held-out
# sample of flows for the demo mode, and (3) proof that the serving code path reproduces the
# test scores reported in the earlier milestones.
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sys.path.append(str(Path("..").resolve()))
from src.evaluation.metrics import evaluate
from src.models.predictor import (
    FEATURES, MODEL_SPECS, classify, load_model, prepare_features, score_against_truth, truth_labels,
)
from src.preprocessing.features import MaxImputer
from src.preprocessing.labels import CLASSES, FAMILY_ENCODING, LABEL_ENCODING, family_labels

PROCESSED = Path("../data/processed")
MODELS = Path("../saved_models")
REPORTS = Path("../reports")
APP_DATA = Path("../app/data")
SEED = 2026
SAMPLE_PER_CLASS = 150

In [2]:
data = pd.read_parquet(PROCESSED / "training_sample.parquet", columns=FEATURES + ["Label", "split_group"])
train = (data["split_group"] == "train").to_numpy()
print(f"rows: {len(data):,}   group-split train: {train.sum():,}   test: {(~train).sum():,}")

rows: 3,598,302   group-split train: 2,935,955   test: 662,347


In [3]:
# The network saw features imputed with the training maximum and standardized with training
# mean/std, both fitted on the group-split TRAIN rows only. Refit them exactly that way.
preprocessor = Pipeline([("impute", MaxImputer()), ("scale", StandardScaler())]).fit(data.loc[train, FEATURES])
joblib.dump(preprocessor, MODELS / "hybrid_preprocessor_group.joblib")
print("saved", MODELS / "hybrid_preprocessor_group.joblib")

saved ..\saved_models\hybrid_preprocessor_group.joblib


In [4]:
# Demo flows: a class-balanced draw from the group-split TEST rows, so the served (group-trained)
# models have never seen these behaviors. Same layout as a user upload: the 60 features + Label.
rng = np.random.default_rng(SEED)
test_idx = np.flatnonzero(~train)
labels = data["Label"].to_numpy()[test_idx]
picked = np.concatenate([
    rng.choice(test_idx[labels == c], size=SAMPLE_PER_CLASS, replace=False) for c in CLASSES
])
sample = data.iloc[np.sort(picked)][FEATURES + ["Label"]].reset_index(drop=True)

APP_DATA.mkdir(parents=True, exist_ok=True)
sample.to_csv(APP_DATA / "sample_flows.csv", index=False)
print(f"{len(sample):,} flows, {(APP_DATA / 'sample_flows.csv').stat().st_size / 1e6:.1f} MB")
sample["Label"].value_counts().to_frame("flows").T

1,950 flows, 0.7 MB


Label,BENIGN,DNS,LDAP,MSSQL,NetBIOS,NTP,SNMP,SSDP,UDP,Syn,TFTP,UDPLag,Portmap
flows,150,150,150,150,150,150,150,150,150,150,150,150,150


In [5]:
# The serving path (prepare_features -> preprocessor -> model) must reproduce, on the full group-split
# test set, the scores recorded when each model was trained. If it does not, the app would be
# serving something different from what the report evaluates.
test_X = prepare_features(data.loc[~train, FEATURES])
y_13 = data.loc[~train, "Label"].map(LABEL_ENCODING).astype("int64").to_numpy()
y_8 = family_labels(data.loc[~train, "Label"]).map(FAMILY_ENCODING).astype("int64").to_numpy()

reported = {
    "xgb13": pd.read_csv(REPORTS / "baseline_results.csv").query("protocol == 'group' and model == 'XGBoost'"),
    "xgb8": pd.read_csv(REPORTS / "family_baseline_results.csv").query("split == 'group'"),
    "hybrid13": pd.read_csv(REPORTS / "hybrid_results.csv").query("split == 'group'"),
}

rows = []
for key, spec in MODEL_SPECS.items():
    proba = load_model(key).predict_proba(test_X)
    y_true = y_8 if spec.level == "attack family" else y_13
    summary, _ = evaluate(y_true, proba.argmax(axis=1), list(spec.classes))
    rows.append({"model": key, "reported macro-F1": reported[key]["macro_f1"].iloc[0],
                 "served macro-F1": summary["macro_f1"], "served accuracy": summary["accuracy"]})

check = pd.DataFrame(rows).set_index("model")
check["difference"] = check["served macro-F1"] - check["reported macro-F1"]
assert check["difference"].abs().max() < 2e-3, f"serving path does not reproduce reported scores:\n{check}"
check.round(4)

,reported macro-F1,served macro-F1,served accuracy,difference
model,,,,
xgb13,0.7828,0.7828,0.8016,0.0
xgb8,0.9748,0.9748,0.9731,0.0
hybrid13,0.6900,0.6900,0.7183,0.0


In [6]:
# What the app's demo mode will show: the saved sample, read back from CSV exactly as an upload
# would be, classified by each served model and scored against its Label column.
sample_back = pd.read_csv(APP_DATA / "sample_flows.csv")
X_sample = prepare_features(sample_back)

rows = []
for key, spec in MODEL_SPECS.items():
    result = classify(load_model(key), X_sample)
    scored = score_against_truth(truth_labels(sample_back, spec.level), result.predicted, list(spec.classes))
    rows.append({"model": key, "flows": scored["n_labeled"], "accuracy": scored["accuracy"], "macro-F1": scored["macro_f1"]})
pd.DataFrame(rows).set_index("model").round(3)

,flows,accuracy,macro-F1
model,,,
xgb13,1950,0.780,0.772
xgb8,1950,0.972,0.972
hybrid13,1950,0.692,0.680
